In [16]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os

In [17]:
# Parameters
N = 100  # Number of neurons
dt = 0.01  # Time step
T = 10  # Total simulation time (short time)
time_steps = int(T / dt)
n_trials = 10  # Number of trials for averaging
J_values = np.linspace(0, 8, 20)  # Range of J values
p = 0.1  # Connection probability for Erdos-Renyi network

In [18]:
# Helper functions
def sigmoid(x):
    return 1 / (1 + np.exp(-x)) - 0.5

def euler_method(r, A, dt, J):
    inputs = np.dot(A, r)
    return r + dt * (-r + sigmoid(inputs))

def compute_mean_firing_rate(rates):
    return np.mean(rates, axis=1)

def compute_variance_firing_rate(rates):
    return np.var(rates, axis=1)

def process_dataset(folder_path):
    graphml_files = [f for f in os.listdir(folder_path) if f.endswith('.graphml')]
    graphs = []
    for file in graphml_files:
        file_path = os.path.join(folder_path, file)
        graph = nx.read_graphml(file_path)
        graphs.append(graph)

    N = len(graphs[0].nodes)

In [19]:
folder_path_83 = '/workspaces/ESN-unofficial/data/folder_path_83'
folder_path_129 = '/workspaces/ESN-unofficial/data/folder_path_129'
folder_path_234 = '/workspaces/ESN-unofficial/data/folder_path_234'
folder_path_463 = '/workspaces/ESN-unofficial/data/folder_path_463'
folder_path_1015 = '/workspaces/ESN-unofficial/data/folder_path_1015'
averaged_1015 = '/workspaces/ESN-unofficial/data/folder_path_1015/averaged_1015'

In [20]:
folder_paths = [folder_path_83, folder_path_129]

for folder_path in folder_paths:
    process_dataset(folder_path)

graphml_files = [f for f in os.listdir(folder_path) if f.endswith('.graphml')]
graphs = []
for file in graphml_files:
     file_path = os.path.join(folder_path, file)
     graph = nx.read_graphml(file_path)
     graphs.append(graph)

In [21]:
def create_average_matrix(J):
    matrices = []
    for graph in graphs:
        adj_matrix = nx.adjacency_matrix(graph, weight="d11").todense()
        avg_fibers = np.mean(adj_matrix[adj_matrix > 0])  # normalise by average fibers
        scaled_adj_matrix = J * adj_matrix / avg_fibers  # scale and normalise
        matrices.append(scaled_adj_matrix)

    return np.mean(matrices, axis=0)

In [22]:
def create_gaussian_random_network(J):
    return np.random.normal(0, J**2 / N, size=(N, N))

#Assuming `average_matrix` is already computed from the data
def create_data_network(J):
     return create_average_matrix * J

network_creators = {
    "Gaussian Random": create_gaussian_random_network,
    "Data Network": create_data_network,
}

# Storage for results
results = {network_name: {"mean_activity": [], "var_activity": []} for network_name in network_creators.keys()}

# Iterate over network types and J values
for network_name, create_network in network_creators.items():
    for J in J_values:
        trial_means = []
        trial_vars = []

        for _ in range(n_trials):
            # Create the network adjacency matrix
            A = create_network(J)

            # Initialize rates
            r = np.random.uniform(0, 1, N)
            rates = np.zeros((time_steps, N))

            # Simulate dynamics
            for t in range(time_steps):
                rates[t] = r
                r = euler_method(r, A, dt, J)

            # Compute mean and variance at the final time step
            trial_means.append(np.mean(rates[-1]))
            trial_vars.append(np.var(rates[-1]))

        # Average over trials
        results[network_name]["mean_activity"].append(np.mean(trial_means))
        results[network_name]["var_activity"].append(np.mean(trial_vars))

# Plotting results
for network_name in network_creators.keys():
    mean_activity = results[network_name]["mean_activity"]
    var_activity = results[network_name]["var_activity"]

    plt.figure(figsize=(12, 5))

    # Mean activity vs J
    plt.subplot(1, 2, 1)
    plt.plot(J_values, mean_activity, marker='o')
    plt.title(f'Mean Activity vs J ({network_name})')
    plt.xlabel('J (Scaling Factor)')
    plt.ylabel('Mean Activity')

    # Variance vs J
    plt.subplot(1, 2, 2)
    plt.plot(J_values, var_activity, marker='o', color='r')
    plt.title(f'Variance vs J ({network_name})')
    plt.xlabel('J (Scaling Factor)')
    plt.ylabel('Variance')

    plt.tight_layout()
    plt.show()

TypeError: unsupported operand type(s) for *: 'function' and 'float'